# Compile PGF Hedge Share Class Sheet

### Inputs
\\\\\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\Daily\py_reports.xlsx "hdgs" tab

### Files and Folders
\\\\\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\PGF UCITS Share Class Hedges

In [1]:
# define the report function
def pgf_check():
    import time
    start_time_pgf = time.time()
    
    # libraries, libraries!
    import schedule, os
    import pandas as pd
    from pathlib import Path
    from datetime import datetime
    from constants import pthPy, pth_dl, pthHdg, pg_do_nb, pg_co_nb
    from utilities import timediff, prior_working_day
    
    # get report date
    df       = pd.read_excel(pthPy, sheet_name='arc', usecols=[6,7,8]).dropna(subset = ["pgf: PAR-N"])
    k        = df.iloc[1,2]
    rptDate  = k.date() if isinstance(k, datetime) and not pd.isna(k) else prior_working_day(datetime.today()).date()
    
    # construct file names
    UTs_name = os.path.join(pth_dl, f'UTPS PGF_UT_prices({len(df["pgf: UT prices"].dropna())}) {rptDate.strftime("%d%b%Y")}.csv')
    NAV_name = os.path.join(pth_dl,  f'PARN PGF_Holdings({len(df["pgf: PAR-N"    ].dropna())}) {rptDate.strftime("%d%b%Y")}.csv')
    
    filename = pthHdg + fr'\{rptDate.strftime("%Y%m%d")} PGF Share Class Hedges.xlsx'
    if os.path.isfile(filename):
        print(f'{datetime.now().strftime("%Hh%M:%Ss %a %d %b %Y")}: \
{filename.removeprefix(pthHdg)} for {rptDate.strftime("%A %d %B %Y")} \
was completed at {time.ctime(os.path.getmtime(filename))}')
        # https://www.geeksforgeeks.org/python-os-path-getmtime-method/
        return
    else:
        try:
            %run "$pg_do_nb" # downloading
            if os.path.isfile(UTs_name) and os.path.isfile(NAV_name):
                %run "$pg_co_nb" # compiling; can yield 'FileNotFoundError ... UTPS PGF_UT_prices(7) 05Mar2025.csv'
            else:
                if os.path.isfile(NAV_name):
                    print(f"Missing the UT prices file: {UTs_name}")
                else:
                    print(f"Missing the holdings file:  {NAV_name}")

            # # open the reporting folder for review
            # os.startfile(os.path.realpath(pthHdg))
        
        except Exception as e:
            print(e)

        print("\n", f"{timediff(start_time_pgf, time.time())} roundtrip \
time complete the {rptDate.strftime('%a %d %b %Y')} pgf report")

In [2]:
# list the frequency of runs

from datetime import datetime, timedelta

# list of times every 15 minutes from 12:55 PM to 16:55 PM
start = datetime.strptime("12:22", "%H:%M")
end = datetime.strptime("18:30", "%H:%M")
increment = 15   # every 15 minutes

times = []
current = start
while current <= end:
    times.append(current.strftime("%H:%M"))
    current += timedelta(minutes=increment)

print(f"Scheduled run times with a {increment} minute increment:\n {(', ').join(times)}")

Scheduled run times with a 15 minute increment:
 12:22, 12:37, 12:52, 13:07, 13:22, 13:37, 13:52, 14:07, 14:22, 14:37, 14:52, 15:07, 15:22, 15:37, 15:52, 16:07, 16:22, 16:37, 16:52, 17:07, 17:22, 17:37, 17:52, 18:07, 18:22


In [ ]:
# scheduler    https://pypi.org/project/schedule/
import schedule, time

for t in times:
    schedule.every().monday.   at(t).do(lambda: pgf_check())
    schedule.every().tuesday.  at(t).do(lambda: pgf_check())
    schedule.every().wednesday.at(t).do(lambda: pgf_check())
    schedule.every().thursday. at(t).do(lambda: pgf_check())
    schedule.every().friday.   at(t).do(lambda: pgf_check())

while True:
    schedule.run_pending()
    time.sleep(10)



###############################
#                             #
# START pgf_downloading.ipynb #
#                             #
###############################


Getting input variables ...
Mon 07 Sep 2026 PGF files to be downloaded:
  C:\Users\hilton.netta\Downloads\UTPS PGF_UT_prices(7) 07Sep2026.csv
  C:\Users\hilton.netta\Downloads\PARN PGF_Holdings(11) 07Sep2026.csv

 Report date:          Mon 07 Sep 2026 
 Class NAV codes:      PGPCEMD,PGPCGED,PGPGBFD,PGPGIFE,PGPGIFI,PGPRFE,PGPRFG 
 Class holdings codes: PGPRUSD,PGPRZAR,PGPRUS0%,PGPRGBP,PGIPZAR,PGIP_I,PGPCZAR,PGBEUR,PGBGBP,PGBZAR,PGEMZAR

4.0sec getting input variables 

Getting the unit trust prices report ...
... downloading PGF unit trust prices
(2) loading libraries

Expected file name based on input values:
   UTPS PGF_UT_prices(7) 07Sep2026.csv
which does not yet exist in the Downloads folder.

(2a) Checking if rpt_type is 'fnav', in which case, replacing '_C' in fund name
(3) setting report suffix for .xls vs .csv
(4) Im

In [ ]:
# !jupyter nbconvert --to script scheduled_pgf_checker.ipynb # to save this notebook as a .py file